# Drought Risk Notebook (NDVI JSON → Open‑Meteo → Drought Score)

This notebook shows how to:
1) Load your **NDVI JSON** output (`ndvi_output.json`)  
2) Fetch **Open‑Meteo** daily weather for **2025**  
3) Build drought features (30‑day rolling precipitation/temperature/ET₀)  
4) Run an **explainable drought scoring** function and export `drought_output.json`

> Works even if you don’t have NDVI JSON yet: a fallback cell creates a small dummy payload.


In [1]:
# ✅ Install dependencies (Colab/Jupyter)
import sys, subprocess

pkgs = ["requests", "numpy"]
for p in pkgs:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", p])

print("✅ Installed")

✅ Installed


In [3]:
import json
import numpy as np
import requests

## 1) Load NDVI JSON

Place `ndvi_output.json` in the same folder as this notebook (or change the path below).

Expected fields:
- `ndvi_delta`
- `later_scene.valid_pixel_ratio`
- `aoi_bbox_epsg4326` (west, south, east, north)


In [5]:
NDVI_JSON_PATH = "ndvi_output.json"

try:
    with open(NDVI_JSON_PATH, "r") as f:
        ndvi_payload = json.load(f)
    print("✅ Loaded", NDVI_JSON_PATH)
except FileNotFoundError:
    ndvi_payload = None
    print("⚠️ ndvi_output.json not found. Run the next cell to create a dummy payload for testing.")

✅ Loaded ndvi_output.json


In [6]:
# (Optional) Create a dummy NDVI payload if you haven't generated ndvi_output.json yet
if ndvi_payload is None:
    ndvi_payload = {
        "aoi_bbox_epsg4326": [-0.195, 51.490, -0.165, 51.510],
        "later_scene": {"valid_pixel_ratio": 0.82, "date": "2025-08-17", "id": "dummy_later"},
        "earlier_scene": {"valid_pixel_ratio": 0.88, "date": "2025-03-01", "id": "dummy_earlier"},
        "ndvi_delta": -0.07
    }
    print("✅ Created dummy ndvi_payload")

In [7]:
# Extract inputs for drought model
ndvi_delta = ndvi_payload.get("ndvi_delta")
ndvi_valid_pixel_ratio = float(ndvi_payload["later_scene"]["valid_pixel_ratio"])

west, south, east, north = ndvi_payload["aoi_bbox_epsg4326"]
lat = (south + north) / 2.0
lon = (west + east) / 2.0

print("ndvi_delta:", ndvi_delta)
print("ndvi_valid_pixel_ratio:", ndvi_valid_pixel_ratio)
print("AOI centroid lat/lon:", lat, lon)

ndvi_delta: -0.03574979305267334
ndvi_valid_pixel_ratio: 1.0
AOI centroid lat/lon: 51.483000000000004 -0.1265


## 2) Fetch Open‑Meteo Daily Weather (2025)

We use the Open‑Meteo archive endpoint for daily:
- precipitation_sum (mm/day)
- temperature_2m_mean (°C/day)
- et0_fao_evapotranspiration (mm/day)  (if available)

If ET₀ is missing, the drought score still works (with slightly reduced signal).


In [8]:
def fetch_openmeteo_daily(lat, lon, start_date="2025-01-01", end_date="2025-12-31", timezone="GMT"):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "daily": "precipitation_sum,temperature_2m_mean,et0_fao_evapotranspiration",
        "timezone": timezone
    }
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    data = r.json()
    return data.get("daily", {})

daily = fetch_openmeteo_daily(lat, lon)
print("Keys:", daily.keys())
print("Days:", len(daily.get("time", [])))
print("Example day:", daily.get("time", ["?"])[0])

Keys: dict_keys(['time', 'precipitation_sum', 'temperature_2m_mean', 'et0_fao_evapotranspiration'])
Days: 365
Example day: 2025-01-01


## 3) Build drought features (rolling 30‑day) and simple baseline stats

For MVP, we compute a within‑year baseline (2025‑to‑date) rather than multi‑year climatology.


In [9]:
def rolling_sum(arr, window):
    arr = np.asarray(arr, dtype=np.float32)
    out = np.full_like(arr, np.nan, dtype=np.float32)
    for i in range(len(arr)):
        j = max(0, i - window + 1)
        out[i] = np.nansum(arr[j:i+1])
    return out

def rolling_mean(arr, window):
    arr = np.asarray(arr, dtype=np.float32)
    out = np.full_like(arr, np.nan, dtype=np.float32)
    for i in range(len(arr)):
        j = max(0, i - window + 1)
        out[i] = np.nanmean(arr[j:i+1])
    return out

def build_weather_features(daily, day_index=-1):
    ppt = np.asarray(daily["precipitation_sum"], dtype=np.float32)
    tmean = np.asarray(daily["temperature_2m_mean"], dtype=np.float32)
    et0 = np.asarray(daily.get("et0_fao_evapotranspiration", [np.nan]*len(ppt)), dtype=np.float32)

    ppt_30d_series = rolling_sum(ppt, 30)
    tmean_30d_series = rolling_mean(tmean, 30)
    et0_30d_series = rolling_sum(et0, 30)

    i = day_index
    ppt_30d = float(ppt_30d_series[i])
    tmean_30d = float(tmean_30d_series[i])
    et0_30d = float(et0_30d_series[i]) if np.isfinite(et0_30d_series[i]) else None
    wb_30d = float(ppt_30d - et0_30d) if et0_30d is not None else None

    mu_p = float(np.nanmean(ppt_30d_series[:i+1]))
    sd_p = float(np.nanstd(ppt_30d_series[:i+1])) + 1e-6
    mu_t = float(np.nanmean(tmean_30d_series[:i+1]))
    sd_t = float(np.nanstd(tmean_30d_series[:i+1])) + 1e-6

    missing_precip_rate = float(np.mean(~np.isfinite(ppt)))
    missing_temp_rate = float(np.mean(~np.isfinite(tmean)))
    missing_et0_rate = float(np.mean(~np.isfinite(et0)))

    return {
        "ppt_30d": ppt_30d,
        "tmean_30d": tmean_30d,
        "et0_30d": et0_30d,
        "wb_30d": wb_30d,
        "ppt_30d_mu": mu_p,
        "ppt_30d_sigma": sd_p,
        "tmean_30d_mu": mu_t,
        "tmean_30d_sigma": sd_t,
        "missing_precip_rate": missing_precip_rate,
        "missing_temp_rate": missing_temp_rate,
        "missing_et0_rate": missing_et0_rate,
    }

weather_features = build_weather_features(daily, day_index=-1)
weather_features

/tmp/ipykernel_421/381158851.py:32: RuntimeWarning: Mean of empty slice
  mu_p = float(np.nanmean(ppt_30d_series[:i+1]))
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipykernel_421/381158851.py:34: RuntimeWarning: Mean of empty slice
  mu_t = float(np.nanmean(tmean_30d_series[:i+1]))


{'ppt_30d': 61.400001525878906,
 'tmean_30d': 7.646667003631592,
 'et0_30d': 13.94999885559082,
 'wb_30d': 47.450002670288086,
 'ppt_30d_mu': nan,
 'ppt_30d_sigma': nan,
 'tmean_30d_mu': nan,
 'tmean_30d_sigma': nan,
 'missing_precip_rate': 0.0,
 'missing_temp_rate': 0.0,
 'missing_et0_rate': 0.0}

## 4) Drought scoring function (explainable, MVP-ready)

This produces:
- `drought_score` (0–100)
- `tier` (LOW/MEDIUM/HIGH)
- `confidence`
- `drivers` and `warnings`


In [10]:
def _zscore(x, mu, sigma):
    if sigma is None or sigma == 0 or np.isnan(sigma):
        return 0.0
    return float((x - mu) / sigma)

def _tier(score):
    if score is None:
        return "UNKNOWN"
    if score >= 70:
        return "HIGH"
    if score >= 40:
        return "MEDIUM"
    return "LOW"

def compute_drought_risk(ndvi_delta, ndvi_valid_pixel_ratio, weather_features):
    warnings = []
    drivers = []

    ppt_30d = weather_features.get("ppt_30d")
    tmean_30d = weather_features.get("tmean_30d")
    et0_30d = weather_features.get("et0_30d")
    wb_30d = weather_features.get("wb_30d")

    if ppt_30d is None or tmean_30d is None:
        return {
            "drought_score": None,
            "tier": "UNKNOWN",
            "confidence": "LOW",
            "drivers": ["Missing ppt_30d or tmean_30d"],
            "warnings": ["WEATHER_FEATURES_INCOMPLETE"],
        }

    ppt_z = _zscore(float(ppt_30d), float(weather_features.get("ppt_30d_mu", ppt_30d)), float(weather_features.get("ppt_30d_sigma", 1.0)))
    t_z   = _zscore(float(tmean_30d), float(weather_features.get("tmean_30d_mu", tmean_30d)), float(weather_features.get("tmean_30d_sigma", 1.0)))

    ppt_risk  = float(np.clip(-ppt_z, 0, 3) / 3.0)   # deficit => higher risk
    temp_risk = float(np.clip(t_z,   0, 3) / 3.0)    # heat => higher risk

    wb_risk = 0.0
    if wb_30d is not None:
        wb_risk = float(np.clip((-float(wb_30d)) / 50.0, 0, 1))  # tuneable
    elif et0_30d is not None:
        warnings.append("WB_30D_NOT_PROVIDED_COMPUTE_PPT_MINUS_ET0")

    ndvi_risk = 0.0
    if ndvi_delta is None:
        warnings.append("NDVI_DELTA_MISSING")
    else:
        ndvi_risk = float(np.clip((-float(ndvi_delta)) / 0.15, 0, 1))  # tuneable

    score01 = (0.45 * ppt_risk) + (0.30 * ndvi_risk) + (0.15 * temp_risk) + (0.10 * wb_risk)
    drought_score = float(np.clip(score01 * 100.0, 0, 100))

    if ppt_risk > 0.6:
        drivers.append("Low rainfall relative to baseline (30-day precipitation deficit)")
    if temp_risk > 0.6:
        drivers.append("Higher temperatures increasing evaporative demand")
    if wb_risk > 0.6:
        drivers.append("Negative water balance (precipitation < ET₀)")
    if ndvi_risk > 0.6:
        drivers.append("NDVI decline indicates vegetation stress / reduced greenness")
    if not drivers:
        drivers.append("No strong drought signals detected from available features")

    missing_p = float(weather_features.get("missing_precip_rate", 0.0))
    missing_t = float(weather_features.get("missing_temp_rate", 0.0))
    missing_e = float(weather_features.get("missing_et0_rate", 0.0)) if et0_30d is not None else 0.0

    coverage = 1.0 - max(missing_p, missing_t, missing_e)
    conf = min(coverage, float(ndvi_valid_pixel_ratio))

    if ndvi_valid_pixel_ratio < 0.6:
        warnings.append("LOW_NDVI_VALID_PIXEL_RATIO")
    if coverage < 0.9:
        warnings.append("WEATHER_DATA_GAPS")

    confidence = "HIGH" if conf >= 0.8 else "MEDIUM" if conf >= 0.6 else "LOW"

    return {
        "drought_score": drought_score,
        "tier": _tier(drought_score),
        "confidence": confidence,
        "drivers": drivers[:3],
        "warnings": warnings,
        "debug": {
            "ppt_z": float(ppt_z),
            "t_z": float(t_z),
            "ppt_risk": ppt_risk,
            "temp_risk": temp_risk,
            "wb_risk": wb_risk,
            "ndvi_risk": ndvi_risk,
            "ndvi_delta": ndvi_delta,
            "ndvi_valid_pixel_ratio": float(ndvi_valid_pixel_ratio),
        }
    }

drought_result = compute_drought_risk(ndvi_delta, ndvi_valid_pixel_ratio, weather_features)
drought_result

{'drought_score': 7.149958610534668,
 'tier': 'LOW',
 'confidence': 'HIGH',
 'drivers': ['No strong drought signals detected from available features'],
 'warnings': [],
 'debug': {'ppt_z': 0.0,
  't_z': 0.0,
  'ppt_risk': 0.0,
  'temp_risk': 0.0,
  'wb_risk': 0.0,
  'ndvi_risk': 0.23833195368448895,
  'ndvi_delta': -0.03574979305267334,
  'ndvi_valid_pixel_ratio': 1.0}}

## 5) Export drought output JSON (for evidence bundle + pipeline)

This creates `drought_output.json` in your working directory.


In [11]:
with open("drought_output.json", "w") as f:
    json.dump(drought_result, f, indent=2)
print("✅ Saved drought_output.json")

✅ Saved drought_output.json
